## 📦 Imports

In [133]:
# Core libraries
import os
import re
import base64
import requests
from pathlib import Path
from datetime import datetime
from typing import List

# Document processing
from langchain_community.document_loaders import PyPDFLoader, UnstructuredPowerPointLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings and vector store
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# LLM and prompting
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Image processing
import fitz  # PyMuPDF
from PIL import Image
from pptx import Presentation
import io

# PDF generation
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import mm
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak, Table, TableStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
from reportlab.lib import colors

## ⚙️ Configuration

Set up API keys and pipeline parameters.

In [ ]:
# API Configuration
OPENROUTER_API_KEY = "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"
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

# Model Configuration
LLM_MODEL = "x-ai/grok-4.1-fast"
VISION_MODEL = "x-ai/grok-4.1-fast"
EMBEDDING_MODEL = "text-embedding-3-small"

# RAG Configuration
CHUNK_SIZE = 1500  
CHUNK_OVERLAP = 300
MAP_REDUCE_BATCH_SIZE = 10
VECTOR_DB_PATH = "./chroma_db"

# Generation Configuration
TEMPERATURE = 0.2
MAX_TOKENS_TEXT = 8000
MAX_TOKENS_VISION = 2000

# Data paths
DATA_DIR = r"C:\Users\rayen\Desktop\ResumeCour\data"
IMAGE_OUTPUT_DIR = "./extracted_images"

print(f"✓ Configuration loaded")
print(f"  Data directory: {DATA_DIR}")
print(f"  LLM Model: {LLM_MODEL}")
print(f"  Embedding Model: {EMBEDDING_MODEL}")

✓ Configuration loaded
  Data directory: c:\Users\rayen\Desktop\ResumeCour\data
  LLM Model: nvidia/nemotron-nano-12b-v2-vl:free
  Embedding Model: sentence-transformers/all-MiniLM-L6-v2


---

## 📂 Step 1: Document Loading

Load course materials from PDFs and PowerPoint files.

In [144]:
def load_pdf(path: str) -> List[Document]:
    """
    Load PDF files and extract text content.
    
    Args:
        path: Path to a single PDF file or directory containing PDFs
        
    Returns:
        List of Document objects with page content and metadata
    """
    p = Path(path)
    all_documents = []

    if p.is_file():
        pdf_files = [p]
    elif p.is_dir():
        pdf_files = sorted(p.glob("*.pdf"))
    else:
        raise FileNotFoundError(f"No such file or directory: {path}")

    for pdf_file in pdf_files:
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            for doc in documents:
                existing_meta = dict(getattr(doc, "metadata", {}) or {})
                existing_meta['source_file'] = pdf_file.name
                existing_meta['file_type'] = 'pdf'
                doc.metadata = existing_meta

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages from {pdf_file.name}")

        except Exception as e:
            print(f"  ✗ Error loading {pdf_file}: {e}")

    print(f"\n✓ Total PDF documents loaded: {len(all_documents)}")
    return all_documents

In [145]:
def load_powerpoint(path: str) -> List[Document]:
    """
    Load PowerPoint files and extract text content.
    
    Args:
        path: Path to a single PPTX file or directory containing PowerPoint files
        
    Returns:
        List of Document objects with slide content and metadata
    """
    p = Path(path)
    all_documents = []

    if p.is_file():
        pptx_files = [p]
    elif p.is_dir():
        pptx_files = sorted(p.glob("*.pptx")) + sorted(p.glob("*.ppt"))
    else:
        raise FileNotFoundError(f"No such file or directory: {path}")

    for pptx_file in pptx_files:
        try:
            loader = UnstructuredPowerPointLoader(str(pptx_file))
            documents = loader.load()
            
            for doc in documents:
                existing_meta = dict(getattr(doc, "metadata", {}) or {})
                existing_meta['source_file'] = pptx_file.name
                existing_meta['file_type'] = 'powerpoint'
                doc.metadata = existing_meta

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} slides from {pptx_file.name}")

        except Exception as e:
            print(f"  ✗ Error loading {pptx_file}: {e}")

    print(f"\n✓ Total PowerPoint documents loaded: {len(all_documents)}")
    return all_documents

## 🖼️ Step 2: Image Extraction

Extract embedded images from PDF and PowerPoint files for vision analysis.

In [146]:
def extract_images_from_pdf(pdf_path: str, out_dir: str) -> List[str]:
    """
    Extract all embedded images from a PDF file.
    
    Args:
        pdf_path: Path to PDF file
        out_dir: Output directory for extracted images
        
    Returns:
        List of paths to extracted image files
    """
    os.makedirs(out_dir, exist_ok=True)
    doc = fitz.open(pdf_path)
    img_paths = []

    for page_index, page in enumerate(doc):
        for img_index, img in enumerate(page.get_images(full=True)):
            xref = img[0]
            pix = fitz.Pixmap(doc, xref)
            img_name = f"page{page_index}_img{img_index}.png"
            img_path = str(Path(out_dir) / img_name)
            
            if pix.n < 5:
                pix.save(img_path)
            else:
                pix = fitz.Pixmap(fitz.csRGB, pix)
                pix.save(img_path)
                
            img_paths.append(img_path)

    return img_paths

In [147]:
def extract_images_from_pptx(pptx_path: str, out_dir: str) -> List[str]:
    """
    Extract all embedded images from a PowerPoint file.
    
    Args:
        pptx_path: Path to PowerPoint file
        out_dir: Output directory for extracted images
        
    Returns:
        List of paths to extracted image files
    """
    os.makedirs(out_dir, exist_ok=True)
    prs = Presentation(pptx_path)
    img_paths = []
    
    for slide_index, slide in enumerate(prs.slides):
        for shape_index, shape in enumerate(slide.shapes):
            if hasattr(shape, "image"):
                image = shape.image
                image_bytes = image.blob
                img_name = f"slide{slide_index}_img{shape_index}.{image.ext}"
                img_path = str(Path(out_dir) / img_name)
                
                with open(img_path, "wb") as f:
                    f.write(image_bytes)
                
                img_paths.append(img_path)
    
    return img_paths

## 👁️ Step 3: Vision-Based Image Analysis

Analyze extracted images using NVIDIA Nemotron vision model to extract text, formulas, diagrams, and key concepts.

In [148]:
def analyze_image_with_vision(image_path: str) -> str:
    """
    Use vision model to extract comprehensive information from an image.
    
    Extracts:
    - All visible text
    - Mathematical formulas with proper notation
    - Descriptions of diagrams and charts
    - Key concepts and definitions
    - Tables and structured data
    
    Args:
        image_path: Path to image file
    
    Returns:
        Extracted text and analysis of the image content
    """
    with open(image_path, "rb") as img_file:
        image_data = base64.b64encode(img_file.read()).decode('utf-8')
    
    img_format = Path(image_path).suffix.lower().replace('.', '')
    if img_format == 'jpg':
        img_format = 'jpeg'
    
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": VISION_MODEL,
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": """Extract ALL text, formulas, diagrams, and key information from this image.

Provide:
1. All visible text (word-for-word)
2. Mathematical formulas (if any) with proper notation
3. Descriptions of diagrams, charts, or visual elements
4. Key concepts or definitions shown
5. Any tables or structured data

Be extremely detailed and comprehensive - this is for study materials."""
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/{img_format};base64,{image_data}"
                        }
                    }
                ]
            }
        ],
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS_VISION
    }
    
    try:
        response = requests.post(url, json=payload, headers=headers)
        response.raise_for_status()
        result = response.json()
        
        if 'choices' in result and len(result['choices']) > 0:
            return result['choices'][0]['message']['content'].strip()
        else:
            print(f"⚠️ No content extracted from {image_path}")
            return ""
    
    except Exception as e:
        print(f"⚠️ Error analyzing image {image_path}: {e}")
        return ""


def analyze_images_to_documents(img_paths: List[str], source_file: str, file_type: str) -> List[Document]:
    """
    Convert a list of images to Document objects using vision analysis.
    
    Args:
        img_paths: List of image file paths
        source_file: Name of source PDF/PPTX
        file_type: Type of source file ('pdf' or 'powerpoint')
    
    Returns:
        List of Document objects with vision-extracted content
    """
    docs = []
    
    if not img_paths:
        return docs
        
    print(f"\n📸 Analyzing {len(img_paths)} images with vision model...")
    
    for i, img_path in enumerate(img_paths, 1):
        print(f"  Processing image {i}/{len(img_paths)}: {Path(img_path).name}")
        
        text = analyze_image_with_vision(img_path)
        
        if not text:
            continue
        
        docs.append(
            Document(
                page_content=text,
                metadata={
                    "source_file": source_file,
                    "file_type": file_type,
                    "image_path": img_path,
                    "extraction_method": "vision_model"
                },
            )
        )
        
        print(f"    ✓ Extracted {len(text)} characters")
    
    print(f"\n✓ Successfully analyzed {len(docs)} images")
    return docs

## ✂️ Step 4: Text Chunking

Split documents into smaller chunks for efficient embedding and retrieval.

In [150]:
def split_documents(documents: List[Document], chunk_size: int = CHUNK_SIZE, chunk_overlap: int = CHUNK_OVERLAP) -> List[Document]:
    """
    Split documents into smaller chunks for better RAG performance.
    
    Uses recursive character splitting with intelligent separators to maintain
    semantic coherence within chunks.
    
    Strategy:
    - Tries to split on paragraph breaks first (\n\n)
    - Then line breaks (\n)
    - Then sentences (.!?)
    - Then clauses (,;)
    - Finally spaces and characters
    
    Args:
        documents: List of Document objects to split
        chunk_size: Maximum characters per chunk (default: 1500)
        chunk_overlap: Number of overlapping characters between chunks (default: 300)
        
    Returns:
        List of chunked Document objects
    """
    if not documents:
        print("⚠️ No documents to split")
        return []
    
    # Calculate total content size
    total_chars = sum(len(doc.page_content) for doc in documents)
    print(f"\n📊 Document Statistics:")
    print(f"  Total documents: {len(documents)}")
    print(f"  Total characters: {total_chars:,}")
    print(f"  Average doc size: {total_chars // len(documents):,} chars")
    print(f"  Chunk size: {chunk_size} chars")
    print(f"  Chunk overlap: {chunk_overlap} chars")
    print(f"  Expected chunks: ~{total_chars // (chunk_size - chunk_overlap):,}")

    # Better separator hierarchy for academic/course content
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=[
            "\n\n\n",  # Multiple line breaks (section breaks)
            "\n\n",    # Paragraph breaks
            "\n",      # Line breaks
            ". ",      # Sentences
            "! ",      # Exclamations
            "? ",      # Questions
            "; ",      # Semicolons
            ", ",      # Commas
            " ",       # Spaces
            ""         # Characters (last resort)
        ],
        keep_separator=True  # Keep separators to maintain formatting
    )
    
    split_docs = text_splitter.split_documents(documents)
    
    # Detailed output
    print(f"\n✓ Split {len(documents)} documents into {len(split_docs)} chunks")
    print(f"  Splitting ratio: {len(split_docs) / len(documents):.1f}x")
    
    if split_docs:
        chunk_sizes = [len(doc.page_content) for doc in split_docs]
        print(f"  Average chunk size: {sum(chunk_sizes) // len(chunk_sizes):,} chars")
        print(f"  Min chunk size: {min(chunk_sizes):,} chars")
        print(f"  Max chunk size: {max(chunk_sizes):,} chars")
        
        print(f"\n📝 Example chunk:")
        print(f"  Content: {split_docs[0].page_content[:200]}...")
        print(f"  Length: {len(split_docs[0].page_content)} chars")
        print(f"  Metadata: {split_docs[0].metadata}")
    
    return split_docs

### 🔍 Diagnose Document Sizes

Check the size of your loaded documents before chunking.

## 🗄️ Step 5: Vector Store Creation

Create embeddings and store them in ChromaDB for efficient similarity search.

In [152]:
def create_vector_store(documents: List[Document], persist_directory: str = VECTOR_DB_PATH) -> Chroma:
    """
    Create embeddings and build a Chroma vector store from documents.
    
    Uses HuggingFace sentence transformers for embedding generation.
    
    Args:
        documents: List of Document objects to embed
        persist_directory: Directory to save the vector database
        
    Returns:
        Chroma vector store instance
    """
    if not documents:
        raise ValueError("No documents provided to create vector store")
    
    print(f"Creating embeddings using {EMBEDDING_MODEL}...")
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True}
    )
    
    print(f"Building Chroma vector store with {len(documents)} documents...")
    vector_store = Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        persist_directory=persist_directory
    )
    
    print(f"✓ Vector store created and persisted to {persist_directory}")
    return vector_store

In [153]:
def load_vector_store(persist_directory: str = VECTOR_DB_PATH) -> Chroma:
    """
    Load an existing Chroma vector store from disk.
    
    Args:
        persist_directory: Directory containing the persisted vector database
        
    Returns:
        Chroma vector store instance
    """
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True}
    )
    
    vector_store = Chroma(
        persist_directory=persist_directory,
        embedding_function=embeddings
    )
    
    print(f"✓ Vector store loaded from {persist_directory}")
    return vector_store

## 🤖 Step 6: LLM Configuration & Summarization

Configure the LLM and implement map-reduce summarization for generating comprehensive study guides.

In [154]:
# Initialize LLM
llm = ChatOpenAI(
    model=LLM_MODEL,
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS_TEXT
)

print(f"✓ LLM initialized: {LLM_MODEL}")

✓ LLM initialized: nvidia/nemotron-nano-12b-v2-vl:free


In [155]:
def generate_study_guide(docs: List[Document], batch_size: int = MAP_REDUCE_BATCH_SIZE) -> str:
    """
    Generate a comprehensive exam study guide using map-reduce summarization.
    
    This is a token-efficient approach that:
    1. Splits documents into batches (MAP phase)
    2. Extracts key information from each batch
    3. Combines into a final comprehensive study guide (REDUCE phase)
    
    The study guide includes:
    - All definitions, formulas, and concepts
    - Detailed methods and algorithms
    - Worked examples
    - Exam preparation tips
    
    Automatically detects language and responds in the same language (French/English).
    
    Args:
        docs: List of Document objects containing course content
        batch_size: Number of documents per batch
    
    Returns:
        Comprehensive study guide in markdown format
    """
    if not docs:
        return "No documents provided for summarization"
    
    # Calculate target word counts based on document volume
    num_docs = len(docs)
    if num_docs <= 10:
        batch_words = 500
        final_words = 2000
    elif num_docs <= 30:
        batch_words = 700
        final_words = 4000
    elif num_docs <= 50:
        batch_words = 900
        final_words = 6000
    else:
        batch_words = 1200
        final_words = 8000
    
    # MAP PHASE: Extract information from each batch
    map_prompt = ChatPromptTemplate.from_template(
        f"""Extract ALL study-relevant information from these course materials for exam preparation:

**IMPORTANT: Respond in the SAME LANGUAGE as the course content (French → French, English → English)**

Extract and list in detail:

1. **DÉFINITIONS / DEFINITIONS**: Every technical term with complete definition
2. **FORMULES / FORMULAS**: ALL mathematical formulas, equations with:
   - Complete notation (write the actual formula)
   - Variable explanations
   - Numerical examples if available
3. **MÉTHODES / METHODS**: Step-by-step procedures for each algorithm/method
4. **CONCEPTS**: Key theories with explanations
5. **EXEMPLES / EXAMPLES**: Worked problems with solutions
6. **PROPRIÉTÉS / PROPERTIES**: Important characteristics and rules

Target: ~{batch_words} words - BE VERY DETAILED, include every formula and definition!

{{{{context}}}}

Detailed extraction:""")
    
    batch_summaries = []
    total_batches = (len(docs) + batch_size - 1) // batch_size
    
    print(f"\n{'='*80}")
    print(f"📊 MAP PHASE: Processing {len(docs)} documents in {total_batches} batches")
    print(f"{'='*80}")
    
    for i in range(0, len(docs), batch_size):
        batch = docs[i:i + batch_size]
        batch_num = i // batch_size + 1
        
        # Create context for this batch
        context_parts = []
        for doc in batch:
            source = doc.metadata.get('source_file', 'Unknown')
            content = doc.page_content[:5000] if len(doc.page_content) > 5000 else doc.page_content
            context_parts.append(f"[{source}]\n{content}")
        
        context = "\n\n---\n\n".join(context_parts)
        
        # Extract information from batch
        chain = {"context": RunnablePassthrough()} | map_prompt | llm
        response = chain.invoke(context)
        batch_summaries.append(response.content)
        
        print(f"  ✓ Batch {batch_num}/{total_batches} processed ({len(batch)} documents)")
    
    # REDUCE PHASE: Combine into final study guide
    reduce_prompt = ChatPromptTemplate.from_template(
        f"""Create a COMPREHENSIVE EXAM CHEAT SHEET from these batch extractions.

**CRITICAL: Write EVERYTHING in the SAME LANGUAGE as the batch extractions below.**

Target: ~{final_words} words (very detailed exam reference)

**STRUCTURE YOUR CHEAT SHEET EXACTLY LIKE THIS:**

# 📘 TITRE DU COURS / COURSE TITLE
[One clear title line]

---

# 📖 DÉFINITIONS CLÉS / KEY DEFINITIONS

List EVERY term as:
- **Terme/Term**: Définition complète / Complete definition
- **Terme/Term**: Définition complète / Complete definition
[Include ALL technical vocabulary]

---

# 🔢 FORMULES MATHÉMATIQUES / MATHEMATICAL FORMULAS

For EACH formula:
**Nom de la formule / Formula name:**
- Formule: [Write the actual mathematical expression]
- Variables: [Explain each symbol]
- Utilisation: [When/how to use]
- Exemple: [Numerical example if available]

[List ALL formulas from the materials - do NOT skip any!]

---

# 💡 CONCEPTS THÉORIQUES / THEORETICAL CONCEPTS

For each concept:
**Nom du concept / Concept name:**
- Explication détaillée / Detailed explanation
- Propriétés clés / Key properties
- Relations avec autres concepts / Relations to other concepts
- Applications / Applications

---

# ⚙️ MÉTHODES ET ALGORITHMES / METHODS & ALGORITHMS

For each method:
**Nom de la méthode / Method name:**
1. Étape 1 / Step 1
2. Étape 2 / Step 2
[Complete step-by-step procedure]
- Entrée/Input: [What goes in]
- Sortie/Output: [What comes out]
- Exemple: [Concrete example]

---

# 📝 EXEMPLES RÉSOLUS / WORKED EXAMPLES

**Exemple 1 / Example 1:**
- Énoncé / Problem: [Full problem statement]
- Solution: [Complete step-by-step solution]
- Réponse finale / Final answer: [Result]

[Include multiple examples for complex topics]

---

# ⚖️ COMPARAISONS / COMPARISONS

| Concept A | vs | Concept B |
|-----------|-----|-----------|
| Différence 1 | | Différence 1 |
| Avantages / Advantages | | Avantages / Advantages |
| Quand utiliser / When to use | | Quand utiliser / When to use |

---

# ⚠️ POINTS IMPORTANTS À RETENIR / KEY POINTS TO REMEMBER

- ✓ Fait critique 1 / Critical fact 1
- ✓ Fait critique 2 / Critical fact 2
- ⚠️ Erreur courante à éviter / Common mistake to avoid
- 💡 Astuce d'examen / Exam tip

---

# 📋 QUESTIONS TYPES D'EXAMEN / TYPICAL EXAM QUESTIONS

**Questions théoriques / Theoretical questions:**
1. [Example question type]
2. [Example question type]

**Exercices de calcul / Calculation problems:**
1. [Example problem type]
2. [Example problem type]

**CRITICAL INSTRUCTIONS:**
- Extract EVERY SINGLE formula from batch summaries (write the actual mathematical expression)
- Include EVERY definition mentioned
- Provide COMPLETE step-by-step methods
- Add worked examples with full solutions
- DO NOT summarize or compress - students need ALL details for exam
- Use bullet points, numbered lists, and tables for easy reference

{{{{summaries}}}}

Complete Exam Cheat Sheet:""")
    
    combined_summaries = "\n\n=== BATCH EXTRACTION ===\n\n".join(batch_summaries)
    
    print(f"\n{'='*80}")
    print(f"🔄 REDUCE PHASE: Combining {len(batch_summaries)} batch extractions")
    print(f"{'='*80}")
    
    chain = {"summaries": RunnablePassthrough()} | reduce_prompt | llm
    final_summary = chain.invoke(combined_summaries)
    
    word_count = len(final_summary.content.split())
    print(f"\n✓ Study guide generated successfully (~{word_count} words)")
    
    return final_summary.content

## 📄 Step 7: PDF Export

Export the generated study guide to a professionally formatted PDF.

In [156]:
def export_to_pdf(summary_text: str, output_filename: str = "course_summary.pdf") -> str:
    """
    Export the study guide to a beautifully formatted PDF.
    
    Features:
    - Custom typography and colors
    - Section headers with backgrounds
    - Formula highlighting
    - Proper markdown to HTML conversion
    - Professional layout with page breaks
    
    Args:
        summary_text: Markdown-formatted study guide text
        output_filename: Output PDF file path
    
    Returns:
        Path to the generated PDF file
    """
    doc = SimpleDocTemplate(
        output_filename,
        pagesize=A4,
        rightMargin=20*mm,
        leftMargin=20*mm,
        topMargin=25*mm,
        bottomMargin=25*mm
    )
    
    story = []
    styles = getSampleStyleSheet()
    
    # Define custom styles
    title_style = ParagraphStyle(
        'CustomTitle',
        parent=styles['Heading1'],
        fontSize=20,
        textColor=colors.HexColor('#1a237e'),
        spaceAfter=30,
        alignment=TA_CENTER,
        fontName='Helvetica-Bold'
    )
    
    section_style = ParagraphStyle(
        'SectionHeader',
        parent=styles['Heading2'],
        fontSize=16,
        textColor=colors.HexColor('#283593'),
        spaceAfter=12,
        spaceBefore=20,
        fontName='Helvetica-Bold',
        borderWidth=2,
        borderColor=colors.HexColor('#3f51b5'),
        borderPadding=5,
        backColor=colors.HexColor('#e8eaf6')
    )
    
    subsection_style = ParagraphStyle(
        'SubsectionHeader',
        parent=styles['Heading3'],
        fontSize=13,
        textColor=colors.HexColor('#5c6bc0'),
        spaceAfter=8,
        spaceBefore=12,
        fontName='Helvetica-Bold'
    )
    
    definition_style = ParagraphStyle(
        'Definition',
        parent=styles['Normal'],
        fontSize=10,
        leading=14,
        alignment=TA_JUSTIFY,
        leftIndent=10,
        rightIndent=10,
        spaceAfter=8,
        bulletIndent=5
    )
    
    formula_style = ParagraphStyle(
        'Formula',
        parent=styles['Code'],
        fontSize=9,
        leading=12,
        textColor=colors.HexColor('#1b5e20'),
        backColor=colors.HexColor('#f1f8e9'),
        leftIndent=15,
        rightIndent=15,
        spaceAfter=6,
        fontName='Courier'
    )
    
    concept_style = ParagraphStyle(
        'Concept',
        parent=styles['Normal'],
        fontSize=10,
        leading=13,
        alignment=TA_JUSTIFY,
        leftIndent=8,
        spaceAfter=8
    )
    
    # Parse markdown content
    lines = summary_text.split('\n')
    current_section = None
    
    for line in lines:
        line = line.strip()
        
        if not line:
            story.append(Spacer(1, 6))
            continue
        
        # Main title
        if line.startswith('# ') and any(emoji in line for emoji in ['📘', '📖', '🔢', '💡']):
            title_text = re.sub(r'[#📘📖🔢💡]', '', line).strip()
            story.append(Paragraph(title_text, title_style))
            story.append(Spacer(1, 12))
            
        # Section separator
        elif line == '---':
            story.append(Spacer(1, 10))
            story.append(Table([['']], colWidths=[doc.width], 
                             style=[('LINEABOVE', (0,0), (-1,0), 2, colors.HexColor('#9fa8da'))]))
            story.append(Spacer(1, 10))
            
        # Main sections
        elif line.startswith('# '):
            section_text = line.replace('#', '').strip()
            emoji_match = re.match(r'([📖🔢💡⚙️📝⚖️⚠️📋])\s*(.*)', section_text)
            if emoji_match:
                emoji, text = emoji_match.groups()
                section_text = f"{emoji} {text}"
            
            story.append(PageBreak())
            story.append(Paragraph(section_text, section_style))
            current_section = section_text
            
        # Subsections
        elif line.startswith('**') and line.endswith(':**'):
            subsection_text = line.replace('**', '').replace(':', '').strip()
            story.append(Paragraph(subsection_text, subsection_style))
            
        # List items
        elif line.startswith('- '):
            content = line[2:].strip()
            
            if '**' in content or ':' in content:
                formatted = re.sub(r'\*\*([^*]+)\*\*', r'<b>\1</b>', content)
                formatted = formatted.replace('&', '&amp;')
                
                try:
                    para = Paragraph(f"• {formatted}", definition_style)
                    story.append(para)
                except Exception:
                    plain_text = re.sub(r'<[^>]+>', '', formatted)
                    para = Paragraph(f"• {plain_text}", definition_style)
                    story.append(para)
            else:
                para = Paragraph(f"• {content}", concept_style)
                story.append(para)
                
        # Formula blocks
        elif 'Formule:' in line or 'Formula:' in line or ('=' in line and ('mod' in line or '*' in line)):
            formula_text = line.replace('Formule:', '<b>Formula:</b>').replace('Formula:', '<b>Formula:</b>')
            story.append(Paragraph(formula_text, formula_style))
            
        # Keywords
        elif any(keyword in line for keyword in ['Variables:', 'Utilisation:', 'Usage:', 'Example:', 'Exemple:']):
            for keyword in ['Variables:', 'Utilisation:', 'Usage:', 'Example:', 'Exemple:']:
                if keyword in line:
                    line = line.replace(keyword, f'<b>{keyword}</b>')
                    break
            story.append(Paragraph(line, concept_style))
            
        # Regular paragraphs
        else:
            text = re.sub(r'\*\*([^*]+)\*\*', r'<b>\1</b>', line)
            text = re.sub(r'\*([^*]+)\*', r'<i>\1</i>', text)
            
            try:
                if current_section and 'FORMULES' in current_section:
                    para = Paragraph(text, formula_style if '=' in text else concept_style)
                else:
                    para = Paragraph(text, concept_style)
                story.append(para)
            except Exception:
                plain_text = re.sub(r'<[^>]+>', '', text)
                para = Paragraph(plain_text, concept_style)
                story.append(para)
    
    # Footer
    footer_style = ParagraphStyle(
        'Footer',
        parent=styles['Normal'],
        fontSize=8,
        textColor=colors.grey,
        alignment=TA_CENTER
    )
    
    story.append(PageBreak())
    story.append(Spacer(1, 50))
    story.append(Paragraph(
        f"Document generated on {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        footer_style
    ))
    story.append(Paragraph(
        "RAG-Powered Course Summary Generator",
        footer_style
    ))
    
    doc.build(story)
    
    print(f"✓ PDF successfully generated: {output_filename}")
    return output_filename

---

## 🚀 Complete RAG Pipeline

Run the end-to-end pipeline to generate a comprehensive study guide.

In [159]:
def run_rag_pipeline(data_directory: str = DATA_DIR, 
                     include_vision: bool = True,
                     include_pptx: bool = False) -> str:
    """
    Execute the complete RAG pipeline from data loading to PDF generation.
    
    Pipeline steps:
    1. Load PDF (and optionally PowerPoint) files
    2. Extract and analyze images with vision model (optional)
    3. Combine text and vision-extracted content
    4. Split documents into chunks
    5. Create vector store with embeddings
    6. Generate comprehensive study guide using map-reduce
    7. Save to markdown and export to PDF
    
    Args:
        data_directory: Directory containing course materials
        include_vision: Whether to analyze images with vision model
        include_pptx: Whether to include PowerPoint files
        
    Returns:
        Generated study guide text
    """
    print(f"\n{'='*80}")
    print(f"🚀 STARTING RAG PIPELINE")
    print(f"{'='*80}")
    print(f"Data directory: {data_directory}")
    print(f"Vision analysis: {'Enabled' if include_vision else 'Disabled'}")
    print(f"PowerPoint support: {'Enabled' if include_pptx else 'Disabled'}")
    
    # Check if directory exists
    if not os.path.exists(data_directory):
        print(f"\n⚠️ Directory not found. Creating: {data_directory}")
        os.makedirs(data_directory, exist_ok=True)
        return "Please place your PDF files in the data directory and run again."
    
    # Step 1: Load documents
    print(f"\n{'='*80}")
    print(f"📚 STEP 1: LOADING DOCUMENTS")
    print(f"{'='*80}")
    
    pdf_files = list(Path(data_directory).glob("*.pdf"))
    
    if not pdf_files:
        print(f"\n⚠️ No PDF files found in {data_directory}")
        return "Please add PDF files to the data directory and run again."
    
    print(f"Found {len(pdf_files)} PDF file(s)")
    pdf_docs = load_pdf(data_directory)
    
    # Optional: Load PowerPoint files
    pptx_docs = []
    if include_pptx:
        pptx_files = list(Path(data_directory).glob("*.pptx")) + list(Path(data_directory).glob("*.ppt"))
        if pptx_files:
            print(f"\nFound {len(pptx_files)} PowerPoint file(s)")
            pptx_docs = load_powerpoint(data_directory)
    
    # Step 2: Vision analysis (optional)
    vision_docs = []
    if include_vision:
        print(f"\n{'='*80}")
        print(f"👁️ STEP 2: VISION-BASED IMAGE ANALYSIS")
        print(f"{'='*80}")
        
        for pdf_file in pdf_files:
            print(f"\nProcessing images from: {pdf_file.name}")
            images = extract_images_from_pdf(str(pdf_file), IMAGE_OUTPUT_DIR)
            
            if images:
                print(f"  Found {len(images)} images")
                vision_docs.extend(analyze_images_to_documents(images, pdf_file.name, "pdf"))
            else:
                print(f"  No images found")
        
        if include_pptx and pptx_docs:
            for pptx_file in pptx_files:
                print(f"\nProcessing images from: {pptx_file.name}")
                images = extract_images_from_pptx(str(pptx_file), IMAGE_OUTPUT_DIR)
                
                if images:
                    print(f"  Found {len(images)} images")
                    vision_docs.extend(analyze_images_to_documents(images, pptx_file.name, "powerpoint"))
    
    # Combine all documents
    all_docs = pdf_docs + pptx_docs + vision_docs
    
    print(f"\n{'='*80}")
    print(f"📊 DOCUMENT SUMMARY")
    print(f"{'='*80}")
    print(f"  PDF text pages: {len(pdf_docs)}")
    if include_pptx:
        print(f"  PowerPoint slides: {len(pptx_docs)}")
    if include_vision:
        print(f"  Vision-analyzed images: {len(vision_docs)}")
    print(f"  Total documents: {len(all_docs)}")
    
    # Step 3: Split documents
    print(f"\n{'='*80}")
    print(f"✂️ STEP 3: CHUNKING DOCUMENTS")
    print(f"{'='*80}")
    
    chunked_docs = split_documents(all_docs)
    
    # Step 4: Create vector store
    print(f"\n{'='*80}")
    print(f"🗄️ STEP 4: CREATING VECTOR STORE")
    print(f"{'='*80}")
    
    vector_store = create_vector_store(chunked_docs)
    
    # Step 5: Generate study guide
    print(f"\n{'='*80}")
    print(f"🤖 STEP 5: GENERATING STUDY GUIDE")
    print(f"{'='*80}")
    
    summary = generate_study_guide(all_docs)
    
    # Step 6: Save outputs
    print(f"\n{'='*80}")
    print(f"💾 STEP 6: SAVING OUTPUTS")
    print(f"{'='*80}")
    
    # Save markdown
    md_path = "course_summary.md"
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(summary)
    print(f"✓ Markdown saved: {md_path}")
    
    # Export PDF
    pdf_path = export_to_pdf(summary, "course_summary.pdf")
    
    # Final summary
    print(f"\n{'='*80}")
    print(f"✅ PIPELINE COMPLETE")
    print(f"{'='*80}")
    print(f"📄 Markdown: {md_path}")
    print(f"📄 PDF: {pdf_path}")
    print(f"🗄️ Vector DB: {VECTOR_DB_PATH}")
    print(f"\nYour comprehensive exam study guide is ready!")
    
    return summary


# Run the complete pipeline
if __name__ == "__main__":
    summary = run_rag_pipeline(
        data_directory=DATA_DIR,
        include_vision=True,  # Set to False to skip image analysis
        include_pptx=False    # Set to True to include PowerPoint files
    )


🚀 STARTING RAG PIPELINE
Data directory: c:\Users\rayen\Desktop\ResumeCour\data
Vision analysis: Enabled
PowerPoint support: Disabled

📚 STEP 1: LOADING DOCUMENTS
Found 1 PDF file(s)
  ✓ Loaded 24 pages from Chapitre2.pdf

✓ Total PDF documents loaded: 24

👁️ STEP 2: VISION-BASED IMAGE ANALYSIS

Processing images from: Chapitre2.pdf
  ✓ Loaded 24 pages from Chapitre2.pdf

✓ Total PDF documents loaded: 24

👁️ STEP 2: VISION-BASED IMAGE ANALYSIS

Processing images from: Chapitre2.pdf
  Found 32 images

📸 Analyzing 32 images with vision model...
  Processing image 1/32: page0_img0.png
  Found 32 images

📸 Analyzing 32 images with vision model...
  Processing image 1/32: page0_img0.png
    ✓ Extracted 2419 characters
  Processing image 2/32: page0_img1.png
    ✓ Extracted 2419 characters
  Processing image 2/32: page0_img1.png
    ✓ Extracted 1319 characters
  Processing image 3/32: page0_img2.png
    ✓ Extracted 1319 characters
  Processing image 3/32: page0_img2.png
    ✓ Extracted 2157 

### RAG Question-Answering

Use the RAG system to answer specific questions about your course materials.

In [1]:
def answer_question(question: str, k: int = 5) -> str:
    """
    Answer a question using RAG.
    
    Args:
        question: User's question
        k: Number of relevant chunks to retrieve
        
    Returns:
        Answer based on retrieved context
    """
    # Load vector store
    vector_store = load_vector_store(VECTOR_DB_PATH)
    
    # Retrieve relevant context
    retriever = vector_store.as_retriever(search_kwargs={"k": k})
    relevant_docs = retriever.get_relevant_documents(question)
    
    # Build context
    context = "\n\n".join([doc.page_content for doc in relevant_docs])
    
    # Create prompt
    qa_prompt = ChatPromptTemplate.from_template("""
You are a helpful tutor answering questions about course materials.

Use the context below to answer the question. If you cannot answer based on the context, say so.

**IMPORTANT: Respond in the SAME LANGUAGE as the question.**

Context:
{context}

Question: {question}

Answer:""")
    
    # Generate answer
    chain = qa_prompt | llm
    response = chain.invoke({"context": context, "question": question})
    
    return response.content


# Example usage
try:
    question = "Explain the main formulas"  # Change to your question
    
    print(f"Question: {question}")
    print("="*80)
    print("\nSearching knowledge base...")
    
    answer = answer_question(question)
    
    print("\nAnswer:")
    print("="*80)
    print(answer)
    
except Exception as e:
    print(f"⚠️ Error: {e}")
    print("Make sure you've run the pipeline first.")

Question: Explain the main formulas

Searching knowledge base...
⚠️ Error: name 'load_vector_store' is not defined
Make sure you've run the pipeline first.
